# NLP Token Classification (POS Tagging & Chunking)

Clean, working notebook for assignment submission.

In [ ]:
!pip install transformers datasets seqeval -q

In [ ]:

import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import evaluate


In [ ]:

dataset = load_dataset("conll2003")
print(dataset)


In [ ]:

label_list = dataset["train"].features["pos_tags"].feature.names
num_labels = len(label_list)

label2id = {l:i for i,l in enumerate(label_list)}
id2label = {i:l for l,i in label2id.items()}

print(label_list)


In [ ]:

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")


In [ ]:

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []

    for i, label in enumerate(examples["pos_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != prev:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            prev = word_idx

        labels.append(label_ids)

    tokenized["labels"] = labels
    return tokenized


In [ ]:

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)


In [ ]:

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)


In [ ]:

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    evaluation_strategy="epoch"
)


In [ ]:

metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


In [ ]:

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].select(range(2000)),
    eval_dataset=tokenized_dataset["validation"].select(range(500)),
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


In [ ]:
trainer.train()

In [ ]:

results = trainer.evaluate()
print(results)


In [ ]:

def predict(sentence):
    tokens = sentence.split()
    inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)
    outputs = model(**inputs)
    preds = np.argmax(outputs.logits.detach().numpy(), axis=2)

    word_ids = inputs.word_ids()
    result = []
    prev = None

    for i, word_idx in enumerate(word_ids):
        if word_idx is None or word_idx == prev:
            continue
        result.append((tokens[word_idx], label_list[preds[0][i]]))
        prev = word_idx

    return result

print(predict("John works at Google in California"))
